# Factory Scheduling Demo Data Generator — MTO/JIT + OTIF + Batch Splitting

This notebook creates a compact manufacturing scheduling dataset for a CP-SAT scheduler/rescheduler.

The updated business logic is:

- each customer order has an `order_quantity`;
- each order is split into production batches/lots;
- every batch follows the same routing independently;
- the MVP is **MTO / Just-in-Time**, so orders go directly to the customer rather than stock;
- the main KPI is **OTIF**: the full order quantity must be completed by the promised date.

Stock buffers, MTS fulfillment, and Production Wheel / sequence-dependent setup are intentionally outside this MVP.


In [1]:
from __future__ import annotations

import random
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

OUTPUT_DIR = Path("generated_factory_demo_data")
OUTPUT_DIR.mkdir(exist_ok=True)

BASE_DATE = pd.Timestamp("2026-04-20 08:00:00")
TIME_UNIT_MIN = 5  # Keep all generated durations on a 5-minute grid.


## 1. Schema

Important columns added by this version:

- `orders.order_quantity`: total quantity promised to the customer;
- `orders.order_type`: currently always `MTO` for the MVP;
- `orders.promised_date`: business due date for OTIF;
- `operations.batch_id`, `batch_index`, `batch_quantity`: production lot information;
- `operations.operation_quantity`: quantity processed by this operation, equal to `batch_quantity`;
- `operations.unit_processing_time_minutes`: processing time per one unit;
- `operations.processing_time_minutes`: `batch_quantity * unit_processing_time_minutes`;
- `operations.setup_time_minutes`: fixed setup for this lot operation;
- `operations.total_duration_minutes`: duration used by the CP-SAT interval.


In [2]:
MACHINE_GROUPS = {
    "CUT": ["M_CUT_01", "M_CUT_02"],
    "WELD": ["M_WELD_01", "M_WELD_02"],
    "PAINT": ["M_PAINT_01"],
    "ASSY": ["M_ASSY_01", "M_ASSY_02"],
    "PACK": ["M_PACK_01"],
}

PRODUCT_ROUTINGS = {
    "FRAME_A": ["CUT", "WELD", "PAINT", "ASSY", "PACK"],
    "FRAME_B": ["CUT", "WELD", "ASSY", "PACK"],
    "KIT_C": ["CUT", "ASSY", "PACK"],
    "MODULE_D": ["WELD", "PAINT", "ASSY"],
}

CUSTOMER_SEGMENTS = ["OEM", "Distributor", "Rush", "Service"]
PRIORITY_LABELS = {1: "critical", 2: "high", 3: "normal", 4: "low"}


In [3]:
def snap_to_grid(minutes: int) -> int:
    return int(round(minutes / TIME_UNIT_MIN) * TIME_UNIT_MIN)


def generate_machines(machine_groups: Dict[str, List[str]]) -> pd.DataFrame:
    """Create a simple machine master table."""
    rows = []
    for group, machines in machine_groups.items():
        for machine_id in machines:
            rows.append(
                {
                    "machine_id": machine_id,
                    "machine_group": group,
                    "machine_name": machine_id.replace("_", " "),
                    "daily_capacity_minutes": 9 * 60,
                    "efficiency": round(np.random.uniform(0.85, 1.05), 3),
                    "can_preempt": False,
                }
            )
    return pd.DataFrame(rows)


def generate_shifts(machines_df: pd.DataFrame, num_days: int = 3) -> pd.DataFrame:
    """Generate one working shift per machine per day."""
    rows = []
    for _, row in machines_df.iterrows():
        for day in range(num_days):
            shift_start = BASE_DATE + pd.Timedelta(days=day)
            shift_end = shift_start + pd.Timedelta(hours=9)
            rows.append(
                {
                    "machine_id": row["machine_id"],
                    "shift_start": shift_start,
                    "shift_end": shift_end,
                    "is_working": True,
                }
            )
    return pd.DataFrame(rows)


def sample_order_quantity() -> int:
    """Sample MTO order quantity.

    Quantities are large enough to make lot splitting useful, but not so large
    that a single batch operation exceeds one shift.
    """
    values = [6, 8, 10, 12, 15, 18, 20]
    probs = [0.12, 0.18, 0.22, 0.18, 0.14, 0.10, 0.06]
    return int(np.random.choice(values, p=probs))


def split_quantity_into_batches(quantity: int, target_batch_size: int = 5) -> List[int]:
    """Split an order quantity into production lots.

    Example:
        quantity=12, target_batch_size=5 -> [5, 5, 2]
    """
    quantity = int(quantity)
    batches = []
    remaining = quantity
    while remaining > 0:
        batch_qty = min(target_batch_size, remaining)
        batches.append(batch_qty)
        remaining -= batch_qty
    return batches


def sample_unit_processing_time(machine_group: str) -> int:
    """Sample per-unit processing time for one item in an order."""
    ranges = {
        "CUT": (5, 12),
        "WELD": (6, 14),
        "PAINT": (5, 10),
        "ASSY": (7, 16),
        "PACK": (3, 7),
    }
    lo, hi = ranges[machine_group]
    return max(TIME_UNIT_MIN, snap_to_grid(int(np.random.randint(lo, hi + 1))))


def sample_setup_time(machine_group: str) -> int:
    """Sample fixed setup time for one lot operation."""
    ranges = {
        "CUT": (5, 15),
        "WELD": (5, 20),
        "PAINT": (10, 25),
        "ASSY": (5, 15),
        "PACK": (0, 5),
    }
    lo, hi = ranges[machine_group]
    return max(0, snap_to_grid(int(np.random.randint(lo, hi + 1))))


In [4]:
def build_operation_rows_for_order(
    order_id: str,
    routing: List[str],
    quantity: int,
    release_time: pd.Timestamp,
    deadline: pd.Timestamp,
    machine_groups: Dict[str, List[str]],
    target_batch_size: int = 5,
) -> List[dict]:
    """Create routing operations for one MTO order split into production batches.

    Each batch follows the full routing independently. This is the compromise
    between the old "one indivisible order batch" model and a much larger
    "flow of individual units" model.
    """
    rows = []
    batch_quantities = split_quantity_into_batches(quantity, target_batch_size=target_batch_size)

    # Keep per-step technological assumptions stable across batches of the same order.
    step_specs = []
    for sequence_index, group in enumerate(routing, start=1):
        step_specs.append(
            {
                "sequence_index": sequence_index,
                "machine_group": group,
                "unit_processing": sample_unit_processing_time(group),
                "setup": sample_setup_time(group),
                "preferred_machine_id": random.choice(machine_groups[group]),
            }
        )

    for batch_index, batch_quantity in enumerate(batch_quantities, start=1):
        batch_id = f"{order_id}_B{batch_index:02d}"

        for spec in step_specs:
            sequence_index = spec["sequence_index"]
            group = spec["machine_group"]
            unit_processing = spec["unit_processing"]
            processing = batch_quantity * unit_processing
            setup = spec["setup"]
            total_duration = processing + setup

            rows.append(
                {
                    "operation_id": f"{batch_id}_OP_{sequence_index:02d}",
                    "order_id": order_id,
                    "batch_id": batch_id,
                    "batch_index": batch_index,
                    "batch_quantity": batch_quantity,
                    "sequence_index": sequence_index,
                    "machine_group_required": group,
                    "preferred_machine_id": spec["preferred_machine_id"],
                    "operation_quantity": batch_quantity,
                    "unit_processing_time_minutes": unit_processing,
                    "processing_time_minutes": processing,
                    "setup_time_minutes": setup,
                    "total_duration_minutes": total_duration,
                    "release_time": release_time,
                    "deadline": deadline,
                    "promised_date": deadline,
                    "is_outsourcable": bool(np.random.rand() < 0.08),
                }
            )
    return rows


def generate_orders_and_operations(
    num_orders: int,
    machine_groups: Dict[str, List[str]],
    product_routings: Dict[str, List[str]],
    target_batch_size: int = 5,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Generate synthetic MTO orders and batch-split routing operations."""
    order_rows = []
    operation_rows = []

    for idx in range(1, num_orders + 1):
        order_id = f"ORD_{idx:03d}"
        product_family = random.choice(list(product_routings.keys()))
        routing = product_routings[product_family]
        order_quantity = sample_order_quantity()
        priority = int(np.random.choice([1, 2, 3, 4], p=[0.12, 0.23, 0.45, 0.20]))

        # Spread releases over the first 90 minutes to avoid a synchronized start.
        release_offset_slots = int(np.random.randint(0, 18))
        release_time = BASE_DATE + pd.Timedelta(minutes=release_offset_slots * TIME_UNIT_MIN)

        # Generate operation durations before computing the deadline.
        temporary_deadline = release_time
        tentative_ops = build_operation_rows_for_order(
            order_id=order_id,
            routing=routing,
            quantity=order_quantity,
            release_time=release_time,
            deadline=temporary_deadline,
            machine_groups=machine_groups,
            target_batch_size=target_batch_size,
        )

        # This is the total batch workload of the order. Since batches can pipeline,
        # the promised date is not simply this sum; we use it to create a realistic
        # due-date distribution with some tight and some relaxed orders.
        nominal_total = sum(row["total_duration_minutes"] for row in tentative_ops)

        slack_factor = float(np.random.uniform(0.85, 1.35))
        extra_slack = int(np.random.randint(1500, 3000))
        deadline = release_time + pd.Timedelta(minutes=int(nominal_total * slack_factor + extra_slack))
        promised_date = deadline

        for row in tentative_ops:
            row["deadline"] = deadline
            row["promised_date"] = promised_date
        operation_rows.extend(tentative_ops)

        order_rows.append(
            {
                "order_id": order_id,
                "product_family": product_family,
                "order_type": "MTO",
                "order_quantity": order_quantity,
                "target_batch_size": target_batch_size,
                "num_batches": len(split_quantity_into_batches(order_quantity, target_batch_size)),
                "priority": priority,
                "priority_label": PRIORITY_LABELS[priority],
                "release_time": release_time,
                "deadline": deadline,
                "promised_date": promised_date,
                "customer_segment": random.choice(CUSTOMER_SEGMENTS),
            }
        )

    return pd.DataFrame(order_rows), pd.DataFrame(operation_rows)


## 2. Synthetic dataset generation


In [5]:
machines_df = generate_machines(MACHINE_GROUPS)
shifts_df = generate_shifts(machines_df, num_days=5)
orders_df, operations_df = generate_orders_and_operations(
    num_orders=20,
    machine_groups=MACHINE_GROUPS,
    product_routings=PRODUCT_ROUTINGS,
    target_batch_size=5,
)

print("machines:", machines_df.shape)
print("shifts:", shifts_df.shape)
print("orders:", orders_df.shape)
print("operations:", operations_df.shape)
print("average batches per order:", round(orders_df["num_batches"].mean(), 2))
print("total order quantity:", int(orders_df["order_quantity"].sum()))
print("total active work minutes:", int(operations_df["total_duration_minutes"].sum()))

machines_df.head(), orders_df.head(), operations_df.head()


machines: (8, 6)
shifts: (40, 4)
orders: (20, 12)
operations: (212, 17)
average batches per order: 2.75
total order quantity: 241
total active work minutes: 9875


(   machine_id machine_group machine_name  daily_capacity_minutes  efficiency  \
 0    M_CUT_01           CUT     M CUT 01                     540       0.925   
 1    M_CUT_02           CUT     M CUT 02                     540       1.040   
 2   M_WELD_01          WELD    M WELD 01                     540       0.996   
 3   M_WELD_02          WELD    M WELD 02                     540       0.970   
 4  M_PAINT_01         PAINT   M PAINT 01                     540       0.881   
 
    can_preempt  
 0        False  
 1        False  
 2        False  
 3        False  
 4        False  ,
   order_id product_family order_type  order_quantity  target_batch_size  \
 0  ORD_001        FRAME_A        MTO              12                  5   
 1  ORD_002        FRAME_A        MTO              15                  5   
 2  ORD_003        FRAME_A        MTO              12                  5   
 3  ORD_004        FRAME_B        MTO              20                  5   
 4  ORD_005          KI

## 3. Machine downtime scenarios


In [6]:
def generate_downtime_scenarios(
    machines_df: pd.DataFrame,
    scenario_machine_group: str = "WELD",
    event_day: int = 0,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Create a small set of downtime scenarios for replanning experiments."""
    candidate_machines = machines_df.loc[
        machines_df["machine_group"] == scenario_machine_group,
        "machine_id",
    ].tolist()
    machine_id = candidate_machines[0]
    event_start = BASE_DATE + pd.Timedelta(days=event_day, hours=2, minutes=30)

    scenario_rows = [
        {
            "scenario_name": "baseline_no_disruption",
            "description": "Initial MTO/JIT plan without machine stoppage",
            "machine_id": None,
            "event_start": pd.NaT,
            "estimated_duration_minutes": 0,
            "actual_duration_minutes": 0,
        },
        {
            "scenario_name": "optimistic_estimate",
            "description": "Stoppage initially estimated as 5 minutes",
            "machine_id": machine_id,
            "event_start": event_start,
            "estimated_duration_minutes": 5,
            "actual_duration_minutes": 20,
        },
        {
            "scenario_name": "pessimistic_estimate",
            "description": "Stoppage initially estimated as 20 minutes",
            "machine_id": machine_id,
            "event_start": event_start,
            "estimated_duration_minutes": 20,
            "actual_duration_minutes": 20,
        },
        {
            "scenario_name": "updated_after_10_min",
            "description": "Initial estimate 5 min, then updated to 20 min after 10 min",
            "machine_id": machine_id,
            "event_start": event_start,
            "estimated_duration_minutes": 5,
            "actual_duration_minutes": 20,
        },
    ]

    downtime_rows = []
    for i, scenario in enumerate(scenario_rows[1:], start=1):
        downtime_rows.append(
            {
                "event_id": f"DT_{i:03d}",
                "machine_id": scenario["machine_id"],
                "event_start": scenario["event_start"],
                "estimated_duration_minutes": scenario["estimated_duration_minutes"],
                "actual_duration_minutes": scenario["actual_duration_minutes"],
                "reason": "unexpected_micro_stoppage",
                "scenario_name": scenario["scenario_name"],
            }
        )

    return pd.DataFrame(downtime_rows), pd.DataFrame(scenario_rows)


downtime_df, scenarios_df = generate_downtime_scenarios(machines_df)
downtime_df


,event_id,machine_id,event_start,estimated_duration_minutes,actual_duration_minutes,reason,scenario_name
0,DT_001,M_WELD_01,2026-04-20 10:30:00,5,20,unexpected_micro_stoppage,optimistic_estimate
1,DT_002,M_WELD_01,2026-04-20 10:30:00,20,20,unexpected_micro_stoppage,pessimistic_estimate
2,DT_003,M_WELD_01,2026-04-20 10:30:00,5,20,unexpected_micro_stoppage,updated_after_10_min


## 4. Small embedded benchmark instance


In [7]:
def build_embedded_benchmark_instance() -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Build a tiny deterministic MTO benchmark with batch splitting."""
    instance = {
        "B_001": {"quantity": 7, "routing": [("CUT", 8), ("WELD", 12), ("ASSY", 10)]},
        "B_002": {"quantity": 6, "routing": [("WELD", 10), ("CUT", 8), ("PACK", 5)]},
        "B_003": {"quantity": 8, "routing": [("ASSY", 12), ("PAINT", 7), ("PACK", 5)]},
        "B_004": {"quantity": 5, "routing": [("CUT", 10), ("ASSY", 8), ("PACK", 5)]},
        "B_005": {"quantity": 9, "routing": [("WELD", 8), ("PAINT", 8), ("ASSY", 8)]},
        "B_006": {"quantity": 5, "routing": [("CUT", 12), ("WELD", 8), ("PACK", 5)]},
    }

    bench_orders = []
    bench_ops = []
    target_batch_size = 3

    for order_id, spec in instance.items():
        release = BASE_DATE
        quantity = spec["quantity"]
        routing = spec["routing"]
        batches = split_quantity_into_batches(quantity, target_batch_size=target_batch_size)

        nominal = 0
        for batch_qty in batches:
            for _, unit_processing in routing:
                nominal += batch_qty * unit_processing + 5
        deadline = release + pd.Timedelta(minutes=int(nominal * 0.9 + 180))

        bench_orders.append(
            {
                "order_id": order_id,
                "product_family": "BENCHMARK",
                "order_type": "MTO",
                "order_quantity": quantity,
                "target_batch_size": target_batch_size,
                "num_batches": len(batches),
                "priority": 3,
                "priority_label": "normal",
                "release_time": release,
                "deadline": deadline,
                "promised_date": deadline,
                "customer_segment": "benchmark",
            }
        )

        for batch_index, batch_qty in enumerate(batches, start=1):
            batch_id = f"{order_id}_B{batch_index:02d}"
            for sequence_index, (group, unit_processing) in enumerate(routing, start=1):
                setup = 5
                processing = batch_qty * unit_processing
                bench_ops.append(
                    {
                        "operation_id": f"{batch_id}_OP_{sequence_index:02d}",
                        "order_id": order_id,
                        "batch_id": batch_id,
                        "batch_index": batch_index,
                        "batch_quantity": batch_qty,
                        "sequence_index": sequence_index,
                        "machine_group_required": group,
                        "preferred_machine_id": MACHINE_GROUPS[group][0],
                        "operation_quantity": batch_qty,
                        "unit_processing_time_minutes": unit_processing,
                        "processing_time_minutes": processing,
                        "setup_time_minutes": setup,
                        "total_duration_minutes": processing + setup,
                        "release_time": release,
                        "deadline": deadline,
                        "promised_date": deadline,
                        "is_outsourcable": False,
                    }
                )

    bench_orders_df = pd.DataFrame(bench_orders)
    bench_ops_df = pd.DataFrame(bench_ops)
    bench_meta_df = pd.DataFrame(
        [
            {
                "instance_name": "embedded_benchmark_small_mto_otif_batch_split",
                "num_orders": len(bench_orders_df),
                "num_operations": len(bench_ops_df),
                "total_quantity": int(bench_orders_df["order_quantity"].sum()),
            }
        ]
    )
    return bench_orders_df, bench_ops_df, bench_meta_df


bench_orders_df, bench_ops_df, bench_meta_df = build_embedded_benchmark_instance()
bench_orders_df.head(), bench_ops_df.head(), bench_meta_df


(  order_id product_family order_type  order_quantity  target_batch_size  \
 0    B_001      BENCHMARK        MTO               7                  3   
 1    B_002      BENCHMARK        MTO               6                  3   
 2    B_003      BENCHMARK        MTO               8                  3   
 3    B_004      BENCHMARK        MTO               5                  3   
 4    B_005      BENCHMARK        MTO               9                  3   
 
    num_batches  priority priority_label        release_time  \
 0            3         3         normal 2026-04-20 08:00:00   
 1            2         3         normal 2026-04-20 08:00:00   
 2            3         3         normal 2026-04-20 08:00:00   
 3            2         3         normal 2026-04-20 08:00:00   
 4            3         3         normal 2026-04-20 08:00:00   
 
              deadline       promised_date customer_segment  
 0 2026-04-20 14:49:00 2026-04-20 14:49:00        benchmark  
 1 2026-04-20 13:31:00 2026-04-2

## 5. Useful derived features


In [8]:
def build_order_features(operations_df: pd.DataFrame, orders_df: pd.DataFrame) -> pd.DataFrame:
    """Aggregate operation-level data into order-level workload and slack features."""
    duration_col = "total_duration_minutes" if "total_duration_minutes" in operations_df.columns else "processing_time_minutes"
    agg = (
        operations_df.groupby("order_id", as_index=False)
        .agg(
            total_operation_minutes=(duration_col, "sum"),
            num_operations=("operation_id", "count"),
            num_batches=("batch_id", "nunique"),
            num_machine_groups=("machine_group_required", "nunique"),
        )
    )

    out = orders_df.drop(columns=["num_batches"], errors="ignore").merge(agg, on="order_id", how="left")
    out["available_slack_minutes"] = (
        (pd.to_datetime(out["promised_date"]) - pd.to_datetime(out["release_time"])).dt.total_seconds() / 60
        - out["total_operation_minutes"]
    )
    out["deadline_tightness_ratio"] = out["available_slack_minutes"] / out["total_operation_minutes"]
    return out.sort_values(["priority", "promised_date"])


order_features_df = build_order_features(operations_df, orders_df)
order_features_df.head()


,order_id,product_family,order_type,order_quantity,target_batch_size,priority,priority_label,release_time,deadline,promised_date,customer_segment,total_operation_minutes,num_operations,num_batches,num_machine_groups,available_slack_minutes,deadline_tightness_ratio
9,ORD_010,KIT_C,MTO,10,5,2,high,2026-04-20 08:35:00,2026-04-21 17:36:00,2026-04-21 17:36:00,Distributor,220,6,2,3,1761.0,8.004545
17,ORD_018,MODULE_D,MTO,6,5,2,high,2026-04-20 08:45:00,2026-04-21 20:19:00,2026-04-21 20:19:00,Rush,190,6,2,3,1944.0,10.231579
5,ORD_006,KIT_C,MTO,6,5,2,high,2026-04-20 09:10:00,2026-04-22 02:17:00,2026-04-22 02:17:00,Service,200,6,2,3,2267.0,11.335000
8,ORD_009,KIT_C,MTO,6,5,2,high,2026-04-20 08:25:00,2026-04-22 02:46:00,2026-04-22 02:46:00,Rush,200,6,2,3,2341.0,11.705000
15,ORD_016,FRAME_B,MTO,8,5,2,high,2026-04-20 08:20:00,2026-04-22 05:17:00,2026-04-22 05:17:00,OEM,390,8,2,4,2307.0,5.915385


## 6. Consistency checks


In [9]:
def validate_dataset(
    machines_df: pd.DataFrame,
    shifts_df: pd.DataFrame,
    orders_df: pd.DataFrame,
    operations_df: pd.DataFrame,
) -> List[str]:
    """Run lightweight structural checks before export."""
    errors = []

    if machines_df["machine_id"].duplicated().any():
        errors.append("Duplicate machine_id in machines_df")
    if orders_df["order_id"].duplicated().any():
        errors.append("Duplicate order_id in orders_df")
    if operations_df["operation_id"].duplicated().any():
        errors.append("Duplicate operation_id in operations_df")

    required_operation_columns = {"batch_id", "batch_index", "batch_quantity", "operation_quantity"}
    missing_columns = required_operation_columns - set(operations_df.columns)
    if missing_columns:
        errors.append(f"Missing batch-splitting operation columns: {sorted(missing_columns)}")

    missing_orders = set(operations_df["order_id"]) - set(orders_df["order_id"])
    if missing_orders:
        errors.append(f"Operations reference missing orders: {sorted(missing_orders)}")

    known_groups = set(machines_df["machine_group"])
    bad_groups = set(operations_df["machine_group_required"]) - known_groups
    if bad_groups:
        errors.append(f"Unknown machine groups in operations: {sorted(bad_groups)}")

    if (operations_df["operation_quantity"] <= 0).any():
        errors.append("Found non-positive operation quantities")
    if (operations_df["batch_quantity"] <= 0).any():
        errors.append("Found non-positive batch quantities")
    if (operations_df["unit_processing_time_minutes"] <= 0).any():
        errors.append("Found non-positive unit processing times")
    if (operations_df["total_duration_minutes"] <= 0).any():
        errors.append("Found non-positive total durations")

    # Each batch appears once per routing operation, so de-duplicate by order/batch
    # before comparing summed batch quantities to the order quantity.
    batch_qty = (
        operations_df[["order_id", "batch_id", "batch_quantity"]]
        .drop_duplicates()
        .groupby("order_id", as_index=False)["batch_quantity"]
        .sum()
        .rename(columns={"batch_quantity": "sum_batch_quantity"})
    )
    qty_check = orders_df[["order_id", "order_quantity"]].merge(batch_qty, on="order_id", how="left")
    bad_qty = qty_check[qty_check["order_quantity"] != qty_check["sum_batch_quantity"]]
    if not bad_qty.empty:
        errors.append("Some orders have batch quantities that do not sum to order_quantity")

    max_shift_duration = (
        pd.to_datetime(shifts_df["shift_end"]) - pd.to_datetime(shifts_df["shift_start"])
    ).dt.total_seconds().max() / 60
    too_long = operations_df[operations_df["total_duration_minutes"] > max_shift_duration]
    if not too_long.empty:
        errors.append(
            f"{len(too_long)} operations exceed the maximum single-shift duration. "
            "The current solver is non-preemptive and requires each operation to fit into one shift."
        )

    if (pd.to_datetime(orders_df["promised_date"]) < pd.to_datetime(orders_df["release_time"])).any():
        errors.append("Some orders have a promised_date earlier than release_time")

    return errors


validation_errors = validate_dataset(machines_df, shifts_df, orders_df, operations_df)
print("validation_errors:", validation_errors if validation_errors else "none")


validation_errors: none


## 7. Export to CSV


In [10]:
def save_dataset_bundle(
    output_dir: Path,
    machines_df: pd.DataFrame,
    shifts_df: pd.DataFrame,
    orders_df: pd.DataFrame,
    operations_df: pd.DataFrame,
    downtime_df: pd.DataFrame,
    scenarios_df: pd.DataFrame,
    prefix: str = "synthetic_demo",
) -> Dict[str, Path]:
    """Write one dataset bundle to disk and return saved paths."""
    bundle_dir = output_dir / prefix
    bundle_dir.mkdir(parents=True, exist_ok=True)

    paths = {
        "machines": bundle_dir / "machines.csv",
        "shifts": bundle_dir / "shifts.csv",
        "orders": bundle_dir / "orders.csv",
        "operations": bundle_dir / "operations.csv",
        "downtime_events": bundle_dir / "downtime_events.csv",
        "scenarios": bundle_dir / "scenarios.csv",
    }

    machines_df.to_csv(paths["machines"], index=False)
    shifts_df.to_csv(paths["shifts"], index=False)
    orders_df.to_csv(paths["orders"], index=False)
    operations_df.to_csv(paths["operations"], index=False)
    downtime_df.to_csv(paths["downtime_events"], index=False)
    scenarios_df.to_csv(paths["scenarios"], index=False)
    return paths


synthetic_paths = save_dataset_bundle(
    OUTPUT_DIR,
    machines_df,
    shifts_df,
    orders_df,
    operations_df,
    downtime_df,
    scenarios_df,
    prefix="synthetic_demo",
)

benchmark_paths = save_dataset_bundle(
    OUTPUT_DIR,
    machines_df,
    shifts_df,
    bench_orders_df,
    bench_ops_df,
    downtime_df,
    scenarios_df,
    prefix="embedded_benchmark",
)
bench_meta_df.to_csv(OUTPUT_DIR / "embedded_benchmark" / "benchmark_meta.csv", index=False)

synthetic_paths, benchmark_paths


({'machines': PosixPath('generated_factory_demo_data/synthetic_demo/machines.csv'),
  'shifts': PosixPath('generated_factory_demo_data/synthetic_demo/shifts.csv'),
  'orders': PosixPath('generated_factory_demo_data/synthetic_demo/orders.csv'),
  'operations': PosixPath('generated_factory_demo_data/synthetic_demo/operations.csv'),
  'downtime_events': PosixPath('generated_factory_demo_data/synthetic_demo/downtime_events.csv'),
  'scenarios': PosixPath('generated_factory_demo_data/synthetic_demo/scenarios.csv')},
 {'machines': PosixPath('generated_factory_demo_data/embedded_benchmark/machines.csv'),
  'shifts': PosixPath('generated_factory_demo_data/embedded_benchmark/shifts.csv'),
  'orders': PosixPath('generated_factory_demo_data/embedded_benchmark/orders.csv'),
  'operations': PosixPath('generated_factory_demo_data/embedded_benchmark/operations.csv'),
  'downtime_events': PosixPath('generated_factory_demo_data/embedded_benchmark/downtime_events.csv'),
  'scenarios': PosixPath('generat

## 8. Quick data preview


In [11]:
print("Synthetic orders by priority")
display(orders_df["priority_label"].value_counts().rename_axis("priority").reset_index(name="count"))

print("\nSynthetic order quantities")
display(orders_df["order_quantity"].describe())

print("\nBatches per order")
display(orders_df["num_batches"].describe())

print("\nOperations per machine group")
display(operations_df["machine_group_required"].value_counts().rename_axis("machine_group").reset_index(name="count"))

print("\nBatch-split operation preview")
display(operations_df[[
    "operation_id",
    "order_id",
    "batch_id",
    "batch_quantity",
    "sequence_index",
    "machine_group_required",
    "total_duration_minutes",
]].head(12))

print("\nDowntime scenarios")
display(downtime_df)


Synthetic orders by priority


,priority,count
0,high,9
1,normal,7
2,low,4



Synthetic order quantities


count    20.000000
mean     12.050000
std       4.706938
min       6.000000
25%       9.500000
50%      12.000000
75%      12.750000
max      20.000000
Name: order_quantity, dtype: float64


Batches per order


...

,machine_group,count
0,ASSY,55
1,CUT,47
2,PACK,47
3,WELD,40
4,PAINT,23



Batch-split operation preview


,operation_id,order_id,batch_id,batch_quantity,sequence_index,machine_group_required,total_duration_minutes
0,ORD_001_B01_OP_01,ORD_001,ORD_001_B01,5,1,CUT,60
1,ORD_001_B01_OP_02,ORD_001,ORD_001_B01,5,2,WELD,45
2,ORD_001_B01_OP_03,ORD_001,ORD_001_B01,5,3,PAINT,65
3,ORD_001_B01_OP_04,ORD_001,ORD_001_B01,5,4,ASSY,40
4,ORD_001_B01_OP_05,ORD_001,ORD_001_B01,5,5,PACK,30
5,ORD_001_B02_OP_01,ORD_001,ORD_001_B02,5,1,CUT,60
6,ORD_001_B02_OP_02,ORD_001,ORD_001_B02,5,2,WELD,45
7,ORD_001_B02_OP_03,ORD_001,ORD_001_B02,5,3,PAINT,65
8,ORD_001_B02_OP_04,ORD_001,ORD_001_B02,5,4,ASSY,40
9,ORD_001_B02_OP_05,ORD_001,ORD_001_B02,5,5,PACK,30



Downtime scenarios


,event_id,machine_id,event_start,estimated_duration_minutes,actual_duration_minutes,reason,scenario_name
0,DT_001,M_WELD_01,2026-04-20 10:30:00,5,20,unexpected_micro_stoppage,optimistic_estimate
1,DT_002,M_WELD_01,2026-04-20 10:30:00,20,20,unexpected_micro_stoppage,pessimistic_estimate
2,DT_003,M_WELD_01,2026-04-20 10:30:00,5,20,unexpected_micro_stoppage,updated_after_10_min


## 9. Minimal scaffold for solver input


In [12]:
def load_bundle_for_scheduler(bundle_dir: Path) -> Dict[str, pd.DataFrame]:
    """Load a saved dataset bundle into memory for downstream scheduling code."""
    return {
        "machines": pd.read_csv(bundle_dir / "machines.csv"),
        "shifts": pd.read_csv(bundle_dir / "shifts.csv", parse_dates=["shift_start", "shift_end"]),
        "orders": pd.read_csv(bundle_dir / "orders.csv", parse_dates=["release_time", "deadline", "promised_date"]),
        "operations": pd.read_csv(bundle_dir / "operations.csv", parse_dates=["release_time", "deadline", "promised_date"]),
        "downtime_events": pd.read_csv(bundle_dir / "downtime_events.csv", parse_dates=["event_start"]),
        "scenarios": pd.read_csv(bundle_dir / "scenarios.csv", parse_dates=["event_start"]),
    }

# Example:
# bundle = load_bundle_for_scheduler(OUTPUT_DIR / "synthetic_demo")


<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=a0401ead-6d52-4f6a-b46e-cd13069c77d8' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>